# Advanced RAG — PDF Ingestion & Structure-Aware Semantic Chunking

**Book:** *Machine Learning* — Tom M. Mitchell (421 pages)

### What was wrong with the previous version

The old notebook extracted "headings" with a single regex — `^#{1,6}\s+(.+)` —
applied to whatever `pymupdf4llm` decided to mark as a heading. That's the bug.
`pymupdf4llm` promotes **anything bold or large-font** to a markdown heading, so
the "structure" list ended up full of noise that isn't real chapter/section
structure:

- Figure/table captions (`FIGURE 2.1`, `**_FIGURE_** 3.3`) treated as headings
- Random emphasized inline phrases (`**A checkers learning problem:**`) — these
  are bolded phrases *inside* a paragraph, not headings, and the same phrase
  appears more than once
- OCR garbage promoted to heading level (`**~ u l k for estimating training
  values.**`)
- No actual chunking step existed yet — the notebook stopped after printing
  `structure[:50]`

If you split on that heading list, section boundaries land in the wrong
places and "semantic" chunks end up mixing unrelated paragraphs — which is
almost certainly the "chunking is wrong" feeling you had.

### Fixed pipeline

1. Extract PDF → Markdown (unchanged, just cached so it doesn't re-run every time)
2. Clean text (unchanged, condensed)
3. **Filter headings** down to *real* section headings only (numbered sections
   + genuine ALL-CAPS titles), dropping captions and inline emphasis
4. Split the book into sections using only the real headings, keeping each
   section's heading path + page number as metadata
5. **Structure-aware semantic chunking**: sections that are already small stay
   as one chunk; large sections get semantically chunked (LangChain's
   `SemanticChunker`, embedding-based breakpoints) instead of naive
   fixed-length splitting — so a chunk only ends where the *meaning* shifts,
   never mid-thought
6. Save chunks with metadata to `.jsonl` for the retrieval index

A pure whole-book semantic chunker (no structure pre-split) is avoided on
purpose — running sentence-level embeddings over ~1.1M characters with no
boundaries is slow and, worse, can merge content across chapter breaks just
because two paragraphs are lexically similar. Splitting on structure first
keeps each semantic-chunking call scoped to one section, which is faster and
more correct.


In [1]:
import re
import json
from pathlib import Path

RAW_PDF = Path("data/raw/MachineLearningTomMitchell.pdf")
MD_PATH = Path("data/processed/ml_book.md")
CLEAN_PATH = Path("data/processed/clean_ml_book.md")
CHUNKS_PATH = Path("data/processed/chunks.jsonl")

MD_PATH.parent.mkdir(parents=True, exist_ok=True)

## Step 1: Extract PDF → Markdown

Cached — only re-runs the (slow, ~6 min) extraction if the output doesn't already exist.

In [2]:
if MD_PATH.exists():
    print(f"Already extracted, skipping. Found: {MD_PATH}")
else:
    import pymupdf
    import pymupdf4llm
    from tqdm import tqdm

    doc = pymupdf.open(RAW_PDF)
    print(f"Total pages: {len(doc)}")

    all_pages = []
    for page_num in tqdm(range(len(doc)), desc="Extracting PDF", unit="page"):
        page_data = pymupdf4llm.to_markdown(doc, pages=[page_num], page_chunks=True)
        all_pages.extend(page_data)
    doc.close()

    with open(MD_PATH, "w", encoding="utf-8") as f:
        for page_number, page in enumerate(all_pages, start=1):
            f.write(f"\n\n<!-- PAGE {page_number} -->\n\n")
            f.write(page["text"])

    print(f"Saved to: {MD_PATH}")

Already extracted, skipping. Found: data\processed\ml_book.md


In [3]:
text = MD_PATH.read_text(encoding="utf-8")
pages = re.findall(r"<!-- PAGE (\d+) -->", text)

print(f"Page markers found: {len(pages)}")
print("First 10:", pages[:10])
print("Last 10:", pages[-10:])

Page markers found: 421
First 10: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
Last 10: ['412', '413', '414', '415', '416', '417', '418', '419', '420', '421']


In [4]:

print(f"Characters: {len(text):,}")
print(f"Words: {len(text.split()):,}")
print(f"Lines: {len(text.splitlines()):,}")


Characters: 1,134,785
Words: 174,545
Lines: 9,602


## Step 2: Clean text

Same normalization as before (line endings, whitespace, blank-line collapsing), also cached.

In [5]:
if CLEAN_PATH.exists():
    print(f"Already cleaned, skipping. Found: {CLEAN_PATH}")
else:
    text = MD_PATH.read_text(encoding="utf-8")

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\n*\s*(<!-- PAGE \d+ -->)\s*\n*", r"\n\n\1\n\n", text)
    text = re.sub(r" +([,.!?;:])", r"\1", text)

    CLEAN_PATH.write_text(text.strip(), encoding="utf-8")
    print(f"Cleaned characters: {len(text):,}")
    print(f"Saved to: {CLEAN_PATH}")

text = CLEAN_PATH.read_text(encoding="utf-8")
print(f"Characters: {len(text):,} | Words: {len(text.split()):,} | Lines: {len(text.splitlines()):,}")

Already cleaned, skipping. Found: data\processed\clean_ml_book.md
Characters: 1,131,404 | Words: 173,680 | Lines: 8,539


In [6]:
headings = re.findall(
    r"(?m)^(#{1,6})\s+(.+?)\s*$",
    text
)

print(f"Total headings: {len(headings)}\n")

for level, title in headings[:100]:
    print(f"{len(level)} | {title}")

Total headings: 365

1 | Machine Learning
2 | Tom M. Mitchell
3 | **Product Details**
4 | **Editorial Reviews**
1 | **PREFACE**
1 | **ACKNOWLEDGMENTS**
1 | **1.1 WELL-POSED LEARNING PROBLEMS**
1 | **A checkers learning problem:**
2 | Control theory
2 | Philosophy
1 | **A robot driving learning problem:**
1 | **1.2 DESIGNING A LEARNING SYSTEM**
1 | **1.2.1 Choosing the Training Experience**
1 | **A checkers learning problem:**
1 | **1.2.2 Choosing the Target Function**
1 | **1.23 Choosing a Representation for the Target Function**
1 | **1.2.4 Choosing a Function Approximation Algorithm**
1 | **1.2.4.1 ESTIMATING TRAINING VALUES**
1 | **~ u l k for estimating training values.**
1 | **1.2.4.2 ADJUSTING THE WEIGHTS**
2 | **LMS weight update rule.**
1 | **1.2.5 The Final Design**
1 | **1.3 PERSPECTIVES AND ISSUES IN MACHINE LEARNING**
1 | **1.3.1 Issues in Machine Learning**
1 | **1.4 HOW TO READ THIS BOOK**
1 | **1.5 SUMMARY AND FURTHER READING**
1 | **EXERCISES**
1 | **REFERENCES**
1 | **

## Step 3: Filter headings down to real section titles

A line only counts as a real heading if it's:

- a **numbered section**, e.g. `3.7.1 Avoiding Overfitting the Data`, or
- a genuine **ALL-CAPS title** (`PREFACE`, `EXERCISES`, `REFERENCES`, `SUMMARY AND FURTHER READING`, ...)

and it's rejected outright if it's a `FIGURE`/`TABLE` caption. This single
filter removes the captions and inline-bold false positives from the old
`structure` list.


In [7]:
HEADING_RE = re.compile(r"(?m)^(#{1,6})\s+(.+?)\s*$")
PAGE_RE = re.compile(r"<!-- PAGE (\d+) -->")

def strip_md_emphasis(s: str) -> str:
    return re.sub(r"[*_]+", "", s).strip()

def is_real_heading(raw_title: str) -> bool:
    title = strip_md_emphasis(raw_title)

    # Captions are never section headings, regardless of how they're styled
    if re.match(r"^(FIGURE|TABLE)\b", title, re.IGNORECASE):
        return False

    # Numbered section, e.g. "3.7.1 Avoiding Overfitting the Data" or "1.2 DESIGNING..."
    if re.match(r"^\d+(\.\d+){0,4}\s+\S", title):
        return True

    # Standalone ALL-CAPS titles: PREFACE, EXERCISES, REFERENCES, SUMMARY AND FURTHER READING...
    letters_only = re.sub(r"[^A-Za-z]", "", title)
    if len(letters_only) >= 4 and letters_only.isupper():
        return True

    return False

lines = text.splitlines()
current_page = None
structure = []

for line in lines:
    page_match = PAGE_RE.match(line.strip())
    if page_match:
        current_page = int(page_match.group(1))
        continue

    heading_match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line.strip())
    if heading_match:
        level = len(heading_match.group(1))
        raw_title = heading_match.group(2)
        if is_real_heading(raw_title):
            structure.append({
                "page": current_page,
                "level": level,
                "title": strip_md_emphasis(raw_title),
                "line_index": len(structure),  # placeholder, fixed below
            })

print(f"Real section headings kept: {len(structure)}")
for item in structure[:20]:
    print(f"p{item['page']:>3} | {'  ' * (item['level'] - 1)}{item['title']}")

Real section headings kept: 298
p  3 | PREFACE
p  4 | ACKNOWLEDGMENTS
p 14 | 1.1 WELL-POSED LEARNING PROBLEMS
p 17 | 1.2 DESIGNING A LEARNING SYSTEM
p 17 | 1.2.1 Choosing the Training Experience
p 19 | 1.2.2 Choosing the Target Function
p 20 | 1.23 Choosing a Representation for the Target Function
p 21 | 1.2.4 Choosing a Function Approximation Algorithm
p 22 | 1.2.4.1 ESTIMATING TRAINING VALUES
p 22 | 1.2.4.2 ADJUSTING THE WEIGHTS
p 23 | 1.2.5 The Final Design
p 26 | 1.3 PERSPECTIVES AND ISSUES IN MACHINE LEARNING
p 27 | 1.3.1 Issues in Machine Learning
p 28 | 1.4 HOW TO READ THIS BOOK
p 29 | 1.5 SUMMARY AND FURTHER READING
p 30 | EXERCISES
p 31 | REFERENCES
p 32 | 2.1 INTRODUCTION
p 33 | 2.2 A CONCEPT LEARNING TASK
p 34 | 2.2.1 Notation


## Step 4: Split the book into sections using only the real headings

Each section keeps the *full heading path* (e.g. `3 DECISION TREE LEARNING > 3.4 THE BASIC DECISION TREE LEARNING ALGORITHM`) so downstream chunks retain their place in the book, plus the starting page for citation.

In [8]:
def find_heading_line_positions(text: str, structure: list[dict]) -> list[int]:
    # Match each kept heading back to its character offset in `text`, in order.
    positions = []
    search_from = 0
    for item in structure:
        pattern = re.compile(
            r"(?m)^#{1,6}\s+\*{0,3}_{0,3}" + re.escape(item["title"][:40]),
        )
        m = pattern.search(text, search_from)
        if m is None:
            # fallback: search from the start if ordering assumption fails
            m = pattern.search(text)
        positions.append(m.start() if m else search_from)
        if m:
            search_from = m.end()
    return positions

positions = find_heading_line_positions(text, structure)

def heading_path(idx: int) -> str:
    # Build 'Chapter title > Section title' using the nearest lower-level ancestor.
    level = structure[idx]["level"]
    path = [structure[idx]["title"]]
    for j in range(idx - 1, -1, -1):
        if structure[j]["level"] < level:
            path.insert(0, structure[j]["title"])
            level = structure[j]["level"]
        if level <= 1:
            break
    return " > ".join(path)

sections = []
for i, item in enumerate(structure):
    start = positions[i]
    end = positions[i + 1] if i + 1 < len(positions) else len(text)
    body = text[start:end]
    body = re.sub(r"(?m)^#{1,6}\s+.+?$", "", body, count=1).strip()  # drop the heading line itself
    body = PAGE_RE.sub("", body).strip()  # page markers add noise inside section body

    sections.append({
        "title": item["title"],
        "heading_path": heading_path(i),
        "page": item["page"],
        "text": body,
    })

sections = [s for s in sections if s["text"]]
print(f"Sections with body text: {len(sections)}")
print(f"Example: {sections[10]['heading_path']!r} (p{sections[10]['page']}, {len(sections[10]['text'])} chars)")

Sections with body text: 280
Example: '1.2.5 The Final Design' (p23, 6499 chars)


## Step 5: Structure-aware semantic chunking

- Sections shorter than `MIN_CHUNK_CHARS` are merged into the next section
  instead of becoming an orphan mini-chunk (common for short intro
  paragraphs right before a subsection).
- Sections are semantically chunked with `SemanticChunker`, which embeds
  sentences and only cuts where consecutive-sentence similarity drops —
  i.e. where the topic actually shifts.
- Sections longer than `MAX_SEMANTIC_INPUT` are pre-split with
  `RecursiveCharacterTextSplitter` first, purely to keep each `SemanticChunker`
  call over a bounded number of sentences (long chapters would otherwise be
  slow to embed in one shot).
- `bge-small-en-v1.5` (~130MB) is used for the embeddings — small enough to
  run on the 4050's 6GB VRAM alongside everything else, or on CPU if you'd
  rather keep the GPU free.


In [9]:
%pip install -q langchain-text-splitters langchain-experimental langchain-huggingface sentence-transformers pymupdf pymupdf4llm

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},   # switch to "cpu" if you want the GPU free for something else
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

fallback_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

MIN_CHUNK_CHARS = 200
MAX_SEMANTIC_INPUT = 20000

W0815 22:38:41.028000 15916 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\Swapn\AppData\Local\Temp\ipykernel_15916\190617687.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Merge sections that are too short to chunk on their own into the next section
merged_sections = []
carry = ""
for sec in sections:
    text_for_section = (carry + "\n\n" + sec["text"]).strip() if carry else sec["text"]
    if len(text_for_section) < MIN_CHUNK_CHARS:
        carry = text_for_section
        continue
    merged_sections.append({**sec, "text": text_for_section})
    carry = ""
if carry:
    if merged_sections:
        merged_sections[-1]["text"] += "\n\n" + carry
    else:
        merged_sections.append({**sections[-1], "text": carry})

print(f"Sections before merge: {len(sections)} -> after merging short ones: {len(merged_sections)}")

Sections before merge: 280 -> after merging short ones: 280


In [12]:
def chunk_section_text(text: str) -> list[str]:
    pieces = fallback_splitter.split_text(text) if len(text) > MAX_SEMANTIC_INPUT else [text]
    out = []
    for piece in pieces:
        out.extend(semantic_splitter.split_text(piece))
    return out

chunks = []
for sec in merged_sections:
    for piece in chunk_section_text(sec["text"]):
        piece = piece.strip()
        if not piece:
            continue
        chunks.append({
            "chunk_id": len(chunks),
            "heading_path": sec["heading_path"],
            "page": sec["page"],
            "text": piece,
            "n_chars": len(piece),
        })

print(f"Total chunks: {len(chunks)}")
lengths = [c["n_chars"] for c in chunks]
print(f"Chunk length — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths) / len(lengths):.0f}")

Total chunks: 1234
Chunk length — min: 1, max: 6809, avg: 897


## Step 6: Sanity check a few chunks

In [13]:
for c in chunks[1000:1003]:
    print(f"[chunk {c['chunk_id']}] {c['heading_path']} (p{c['page']}, {c['n_chars']} chars)")
    print(c["text"][:400].replace("\n", " "))
    print("-" * 80)

[chunk 1000] 12.1 MOTIVATION (p346, 233 chars)
Statistical justifications are only as compelling as the data and statistical assumptions on which they rest. They are suspect or powerless when assumptions about the underlying distributions cannot be trusted or when data is scarce.
--------------------------------------------------------------------------------
[chunk 1001] 12.1 MOTIVATION (p346, 71 chars)
In short, the two approaches work well for different types of problems.
--------------------------------------------------------------------------------
[chunk 1002] 12.1 MOTIVATION (p346, 1333 chars)
By combining them we can hope to devise a more general learning approach that covers a more broad range of learning tasks. Figure 12.1 summarizes a spectrum of learning problems that varies by the availability of prior knowledge and training data. At one extreme, a large volume   ||**Inductive learning**|**Analytical learning**| |---|---|---| |**Goal:**<br>**Justification:**|**Hypothesis 

## Step 7: Save chunks for indexing

In [14]:
CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

Saved 1234 chunks to data\processed\chunks.jsonl


### Optional next step: graph structure instead of/alongside a flat list

In [15]:
import networkx as nx

G = nx.DiGraph()
for c in chunks:
    parts = c["heading_path"].split(" > ")
    for parent, child in zip(parts, parts[1:]):
        G.add_edge(parent, child)
    G.add_edge(parts[-1], f"chunk_{c['chunk_id']}")
    G.nodes[f"chunk_{c['chunk_id']}"]["text"] = c["text"]

In [16]:
%pip install -q chromadb langchain-chroma

Note: you may need to restart the kernel to use updated packages.


# Creating ChromaDB vector Database

In [17]:
import json
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Load chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

# Convert to LangChain Documents
documents = [
    Document(
        page_content=c["text"],
        metadata={
            "chunk_id": c["chunk_id"],
            "heading_path": c["heading_path"],
            "page": c["page"],
        }
    )
    for c in chunks
]

print(f"Documents: {len(documents)}")

Documents: 1234


In [18]:
CHROMA_PATH = "data/chroma"

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ml_book",
    persist_directory=CHROMA_PATH,
)

print("ChromaDB created successfully.")

ChromaDB created successfully.


In [19]:
results = vectorstore.similarity_search(
    "What is machine learning?",
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:500])
    print(doc.metadata)


--- Result 1 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regularities that can be discovered automatically (e.g., to analyze outcomes of medical treatments from 
{'page': 29, 'heading_path': '1.5 SUMMARY AND FURTHER READING', 'chunk_id': 48}

--- Result 2 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valua

In [20]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [21]:
query = "What is machine learning?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400])
    print("Metadata:", doc.metadata)


--- Result 1 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regu
Metadata: {'heading_path': '1.5 SUMMARY AND FURTHER READING', 'page': 29, 'chunk_id': 48}

--- Result 2 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regu
Metadata: {'heading_path': '1.5 SUMMARY AND FURTHER READING', 'chunk_id'

In [22]:
%pip install -q rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [23]:
from rank_bm25 import BM25Okapi

tokenized_docs = [
    doc.page_content.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

print("BM25 index created.")

BM25 index created.


In [24]:
query = "What is machine learning?"

query_tokens = query.lower().split()

scores = bm25.get_scores(query_tokens)

top_k = 5
top_indices = scores.argsort()[-top_k:][::-1]

for i, idx in enumerate(top_indices, 1):
    print(f"\n--- Result {i} ---")
    print(documents[idx].page_content[:400])


--- Result 1 ---
When studying machine learning it is natural to wonder what general laws may govern machine (and nonmachine) learners. Is it possible to identify classes of learning problems that are inherently difficult or easy, independent of the learning algorithm? Can one characterize the number of training examples necessary or sufficient to assure successful learning?

--- Result 2 ---
Our checkers example raises a number of generic questions about machine learning. The field of machine learning, and much of this book, is concerned with answering questions such as the following: 

- What algorithms exist for learning general target functions from specific training examples? In what settings will particular algorithms converge to the desired function, given sufficient training da

--- Result 3 ---
In this same sense, every other algorithm discussed elsewhere in this book (e.g., BACKPROPAGATION, C4.5) is an eager learning algorithm. Are there important differences in what can be 

In [25]:
def hybrid_search(query, k=5, fetch_k=10):
    # Dense results
    dense_docs = retriever.invoke(query)[:fetch_k]

    # BM25 results
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    bm25_indices = scores.argsort()[-fetch_k:][::-1]

    # RRF scores
    rrf_scores = {}

    for rank, doc in enumerate(dense_docs):
        doc_id = doc.metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    for rank, idx in enumerate(bm25_indices):
        doc_id = documents[idx].metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    # Sort by combined score
    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:k]

    # Return documents
    doc_lookup = {
        doc.metadata["chunk_id"]: doc
        for doc in documents
    }

    return [doc_lookup[doc_id] for doc_id in ranked_ids]

In [26]:
results = hybrid_search(
    "What is machine learning?",
    k=5
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400])


--- Result 1 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regu

--- Result 2 ---
When studying machine learning it is natural to wonder what general laws may govern machine (and nonmachine) learners. Is it possible to identify classes of learning problems that are inherently difficult or easy, independent of the learning algorithm? Can one characterize the number of training examples necessary or sufficient to assure successful learning?

--- Result 3 ---
Our checkers example raises a number of generic questions about machine learning. The field of machine learning, and much of this book, is concerned with answering questions such as th

In [27]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
def rerank(query, docs, top_k=5):
    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for score, doc in ranked[:top_k]]

In [29]:
query = "What is machine learning?"

hybrid_docs = hybrid_search(
    query,
    k=10
)

final_docs = rerank(
    query,
    hybrid_docs,
    top_k=5
)

for i, doc in enumerate(final_docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400])


--- Result 1 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regu

--- Result 2 ---
Our checkers example raises a number of generic questions about machine learning. The field of machine learning, and much of this book, is concerned with answering questions such as the following: 

- What algorithms exist for learning general target functions from specific training examples? In what settings will particular algorithms converge to the desired function, given sufficient training da

--- Result 3 ---
When studying machine learning it is natural to wonder what general laws may govern machine (and nonmachine) learners. Is it possible to identif

In [30]:
def build_context(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

If the answer is not present in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
""")

In [32]:
#%pip install -q langchain-openai


In [33]:
#%pip install -q -U langchain-google-genai

In [34]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env

def rag(query):
    # Retrieve
    hybrid_docs = hybrid_search(query, k=10)

    # Rerank
    final_docs = rerank(query, hybrid_docs, top_k=5)

    # Context
    context = build_context(final_docs)

    # Prompt
    messages = prompt.invoke({
        "context": context,
        "question": query
    })

    # Generate using Gemini
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",   # or "gemini-2.5-pro" for higher quality
        temperature=0,
        google_api_key=os.getenv("GEMINI_API_KEY")  # explicit, since your .env var isn't named GOOGLE_API_KEY
    )
    response = llm.invoke(messages)

    return response, final_docs

In [35]:
response, sources = rag(
    "What is machine learning?"
)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience.


In [36]:
%pip install -q ragas

Note: you may need to restart the kernel to use updated packages.


In [37]:
question = "What is machine learning?"

response, sources = rag(question)

answer = response.content

contexts = [
    doc.page_content
    for doc in sources
]

print("Question:", question)
print("\nAnswer:", answer)
print("\nNumber of contexts:", len(contexts))

Question: What is machine learning?

Answer: Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience.

Number of contexts: 5


In [39]:
%pip install -U langchain-community-chat

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-community-chat (from versions: none)
ERROR: No matching distribution found for langchain-community-chat


In [40]:
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_dict({
    "user_input": [question],
    "response": [answer],
    "retrieved_contexts": [contexts]
})

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

In [ ]:
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:
result = evaluate(
    eval_dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall()
    ],
    llm=evaluator_llm
)

print(result)

In [ ]:
df = result.to_pandas()
df

In [ ]:
questions = [
    "What is machine learning?",
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is reinforcement learning?",
    "What are the main applications of machine learning?"
]

In [ ]:
eval_data = {
    "user_input": [],
    "response": [],
    "retrieved_contexts": []
}

for question in questions:

    response, sources = rag(question)

    eval_data["user_input"].append(question)
    eval_data["response"].append(response.content)
    eval_data["retrieved_contexts"].append(
        [doc.page_content for doc in sources]
    )

In [ ]:
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_dict(eval_data)

print(f"Evaluation samples: {len(eval_dataset)}")

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

result = evaluate(
    eval_dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall()
    ],
    llm=evaluator_llm
)

df = result.to_pandas()

df

In [ ]:
print(df[
    [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall"
    ]
].mean())